# Python Database Programming Exercises: 5 Coding Problems with Solutions

A practice notebook implementing a small Hospital Information System — connecting, parameterized SELECTs, multi-step lookups, and calculated UPDATEs — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-database-programming-exercise-with-solution/), which offers solutions in MySQL, PostgreSQL, and SQLite side by side. This notebook uses **SQLite** (via Python's built-in `sqlite3` module) throughout, since it needs no external database server and lets every exercise run standalone — each explanation notes how the same code differs for MySQL/PostgreSQL.*

---

## Concepts you'll need

This set is a mini hospital-management project using Python's database (DB-API 2.0-style) interface. The source page offers solutions for MySQL, PostgreSQL, and SQLite side by side; this notebook uses **SQLite** throughout (via the built-in `sqlite3` module) so every exercise runs standalone with no external database server required. The same patterns apply directly to MySQL (`mysql.connector`) and PostgreSQL (`psycopg2`) — the differences are noted in each explanation.

- **The connect / cursor / execute / fetch pattern** — nearly identical across all three drivers: `connection = <driver>.connect(...)` opens a connection, `cursor = connection.cursor()` creates a cursor to run commands through, `cursor.execute(sql, params)` runs a query, and `cursor.fetchone()` / `cursor.fetchall()` retrieves results (one row, or every row as a list).
- **Parameterized queries — always use them for any value coming from outside your code.** Never build SQL with an f-string or `+` concatenation, since that opens the door to SQL injection. Placeholder syntax differs by driver: SQLite and many others use `?`, while MySQL's connector and psycopg2 use `%s` — but the underlying safety mechanism (passing values separately as a tuple, `(value,)`, never inline in the query text) is identical everywhere.
- **`cursor.fetchone()` vs. `cursor.fetchall()`** — `fetchone()` returns a single row as a tuple (or `None` if there were no results) — appropriate when a query can match at most one row, like a lookup by primary key. `fetchall()` returns every matching row as a list of tuples, appropriate for queries that can match several rows.
- **Rows as tuples, indexed positionally** — a fetched row is a plain tuple; `row[0]` is the first selected column, `row[1]` the second, and so on, matching the order columns were listed in the `SELECT`.
- **Writes need `connection.commit()`** — `INSERT`/`UPDATE`/`DELETE` statements only take effect permanently once you call `.commit()` on the connection; without it, the change may be silently rolled back or never persisted, depending on the driver's default transaction behavior.
- **Always close connections** — wrapping each operation in `try`/`except` and calling `.close()` in a `finally` block (or a small helper function, as this exercise does) prevents connections from leaking if an error occurs mid-operation.
- **Computing experience from a join date** — `dateutil.relativedelta(today, joining_date).years` gives a precise whole-year difference between two dates, correctly accounting for whether the anniversary has passed yet this year (unlike simply subtracting the year numbers, which can be off by one).

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Connect to the Database and Print Its Version

**Concept:** the connect() / cursor() / execute() / fetchone() pattern

**Problem:** Connect to the database server and print its version.

**Given:**
```
no input — sets up the Hospital and Doctor tables first
```

**Expected Output:**
```
You are connected to SQLite version: ('3.4x.x',)
```

**Hint:** sqlite_version() is SQLite's equivalent of MySQL's version() or PostgreSQL's version() function.

In [ ]:
import sqlite3
import os

# Set up the database and tables fresh, so this notebook is self-contained
if os.path.exists("python_db.db"):
    os.remove("python_db.db")

def get_connection():
    connection = sqlite3.connect('python_db.db')
    return connection

def close_connection(connection):
    if connection:
        connection.close()

def setup_database():
    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE Hospital (
            Hospital_Id INTEGER NOT NULL PRIMARY KEY,
            Hospital_Name TEXT NOT NULL,
            Bed_Count INTEGER NOT NULL
        )
    """)
    cursor.execute("""
        CREATE TABLE Doctor (
            Doctor_Id INTEGER NOT NULL PRIMARY KEY,
            Doctor_Name TEXT NOT NULL,
            Hospital_Id INTEGER NOT NULL,
            Joining_Date TEXT NOT NULL,
            Speciality TEXT NOT NULL,
            Salary INTEGER NOT NULL,
            Experience INTEGER
        )
    """)

    hospitals = [
        (1, 'Mayo Clinic', 200),
        (2, 'Cleveland Clinic', 400),
        (3, 'Johns Hopkins', 1000),
        (4, 'UCLA Medical Center', 1500),
    ]
    cursor.executemany("INSERT INTO Hospital VALUES (?, ?, ?)", hospitals)

    doctors = [
        (101, 'David', 1, '2005-02-10', 'Pediatric', 40000, None),
        (102, 'Michael', 1, '2018-07-23', 'Oncologist', 20000, None),
        (103, 'Susan', 2, '2016-05-19', 'Gynacologist', 25000, None),
        (104, 'Robert', 2, '2017-12-28', 'Pediatric', 28000, None),
        (105, 'Linda', 3, '2004-06-04', 'Gynacologist', 42000, None),
        (106, 'William', 3, '2012-09-11', 'Dermatologist', 30000, None),
        (107, 'Richard', 4, '2014-08-21', 'Gynacologist', 32000, None),
        (108, 'Karen', 4, '2011-10-17', 'Radiologist', 30000, None),
    ]
    cursor.executemany("INSERT INTO Doctor VALUES (?, ?, ?, ?, ?, ?, ?)", doctors)

    connection.commit()
    close_connection(connection)
    print("Hospital and Doctor tables created and populated.")

setup_database()

def read_database_version():
    try:
        connection = get_connection()
        cursor = connection.cursor()
        cursor.execute("select sqlite_version();")
        db_version = cursor.fetchone()
        print("You are connected to SQLite version:", db_version)
        close_connection(connection)
    except (Exception, sqlite3.Error) as error:
        print("Error while getting data", error)

print("\nQuestion 1: Print Database version")
read_database_version()

**Explanation:** sqlite3.connect('python_db.db') opens (or, if it doesn't exist yet, creates) a local database file — no separate server process is needed, unlike MySQL or PostgreSQL, which require connecting to a running database server with a host, port, username, and password. cursor.execute("select sqlite_version();") runs the query, and cursor.fetchone() retrieves the single resulting row as a tuple. Wrapping the whole operation in try/except (Exception, sqlite3.Error) catches both general Python errors and SQLite-specific ones in a single handler, printing a clear message rather than crashing with a raw traceback.

## Exercise 2. Fetch Hospital and Doctor Information by ID

**Concept:** a parameterized SELECT, executed with cursor.execute(query, (param,))

**Problem:** Fetch and print details for a given hospital ID and a given doctor ID.

**Given:**
```
hospital_id = 2, doctor_id = 105
```

**Expected Output:**
```
Hospital Id: 2, Hospital Name: Cleveland Clinic, Bed Count: 400
Doctor Id: 105, Doctor Name: Linda, ...
```

**Hint:** A trailing comma in (hospital_id,) is required — without it, Python would treat the parentheses as grouping, not a tuple.

In [ ]:
import sqlite3

def get_connection():
    return sqlite3.connect('python_db.db')

def close_connection(connection):
    if connection:
        connection.close()

def get_hospital_detail(hospital_id):
    try:
        connection = get_connection()
        cursor = connection.cursor()
        select_query = """select * from Hospital where Hospital_Id = ?"""
        cursor.execute(select_query, (hospital_id,))
        records = cursor.fetchall()
        print("Printing Hospital record")
        for row in records:
            print("Hospital Id:", row[0])
            print("Hospital Name:", row[1])
            print("Bed Count:", row[2])
        close_connection(connection)
    except (Exception, sqlite3.Error) as error:
        print("Error while getting data", error)

def get_doctor_detail(doctor_id):
    try:
        connection = get_connection()
        cursor = connection.cursor()
        select_query = """select * from Doctor where Doctor_Id = ?"""
        cursor.execute(select_query, (doctor_id,))
        records = cursor.fetchall()
        print("Printing Doctor record")
        for row in records:
            print("Doctor Id:", row[0])
            print("Doctor Name:", row[1])
            print("Hospital Id:", row[2])
            print("Joining Date:", row[3])
            print("Specialty:", row[4])
            print("Salary:", row[5])
            print("Experience:", row[6])
        close_connection(connection)
    except (Exception, sqlite3.Error) as error:
        print("Error while getting data", error)

print("Question 2: Read given hospital and doctor details\n")
get_hospital_detail(2)
print()
get_doctor_detail(105)

**Explanation:** The ? in the query string is a placeholder — SQLite's parameter marker syntax (MySQL's connector and psycopg2 use %s instead, but the principle is identical). Passing (hospital_id,) as a separate tuple argument to execute() lets the database driver safely substitute the value, escaping it correctly and preventing SQL injection — this is fundamentally different and much safer than building the query string with an f-string like f"...WHERE Hospital_Id = {hospital_id}". Each fetched row is a plain tuple, so row[0], row[1], etc. access columns by their position in the SELECT * output, matching the table's column definition order.

## Exercise 3. Get Doctors by Specialty and Minimum Salary

**Concept:** a parameterized query with TWO conditions and TWO bound parameters

**Problem:** Fetch all doctors matching a given specialty whose salary is above a given amount.

**Given:**
```
speciality = "Gynacologist", salary = 30000
```

**Expected Output:**
```
Doctors Linda (42000) and Richard (32000), both Gynacologists earning over 30000
```

**Hint:** The parameter tuple (speciality, salary) must list values in the SAME order the ?s appear in the query string.

In [ ]:
import sqlite3

def get_connection():
    return sqlite3.connect('python_db.db')

def close_connection(connection):
    if connection:
        connection.close()

def get_specialist_doctors_list(speciality, salary):
    try:
        connection = get_connection()
        cursor = connection.cursor()
        sql_select_query = """select * from Doctor where Speciality = ? and Salary > ?"""
        cursor.execute(sql_select_query, (speciality, salary))
        records = cursor.fetchall()
        print(f"Printing doctors whose specialty is {speciality} and salary greater than {salary}\n")
        for row in records:
            print("Doctor Id:", row[0])
            print("Doctor Name:", row[1])
            print("Hospital Id:", row[2])
            print("Joining Date:", row[3])
            print("Specialty:", row[4])
            print("Salary:", row[5])
            print("Experience:", row[6], "\n")
        close_connection(connection)
    except (Exception, sqlite3.Error) as error:
        print("Error while getting data", error)

print("Question 3: Get Doctors as per given Speciality\n")
get_specialist_doctors_list("Gynacologist", 30000)

**Explanation:** The query combines two conditions with SQL's AND, and correspondingly needs two placeholders (? and ?) and two values in the parameter tuple. The order matters: (speciality, salary) fills the placeholders left to right, matching their order of appearance in the query text — swapping the tuple's order without also swapping the query would silently compare the wrong types against the wrong columns.

## Exercise 4. Get All Doctors From a Given Hospital, With the Hospital Name

**Concept:** two sequential queries, using one query's result as input to a second

**Problem:** Fetch every doctor working at a given hospital, showing that hospital's name alongside each doctor's record.

**Given:**
```
hospital_id = 2
```

**Expected Output:**
```
Susan and Robert, both listed with 'Hospital Name: Cleveland Clinic' attached
```

**Hint:** The Doctor table only stores a Hospital_Id (a foreign key) — the actual hospital NAME has to come from a separate lookup in the Hospital table.

In [ ]:
import sqlite3

def get_connection():
    return sqlite3.connect('python_db.db')

def close_connection(connection):
    if connection:
        connection.close()

def get_hospital_name(hospital_id):
    try:
        connection = get_connection()
        cursor = connection.cursor()
        select_query = """select * from Hospital where Hospital_Id = ?"""
        cursor.execute(select_query, (hospital_id,))
        record = cursor.fetchone()
        close_connection(connection)
        return record[1]
    except (Exception, sqlite3.Error) as error:
        print("Error while getting data", error)

def get_doctors(hospital_id):
    try:
        hospital_name = get_hospital_name(hospital_id)
        connection = get_connection()
        cursor = connection.cursor()
        sql_select_query = """select * from Doctor where Hospital_Id = ?"""
        cursor.execute(sql_select_query, (hospital_id,))
        records = cursor.fetchall()

        print("Printing Doctors of", hospital_name, "Hospital")
        for row in records:
            print("Doctor Id:", row[0])
            print("Doctor Name:", row[1])
            print("Hospital Id:", row[2])
            print("Hospital Name:", hospital_name)
            print("Joining Date:", row[3])
            print("Specialty:", row[4])
            print("Salary:", row[5])
            print("Experience:", row[6], "\n")
        close_connection(connection)
    except (Exception, sqlite3.Error) as error:
        print("Error while getting doctor's data", error)

print("Question 4: Get List of doctors of a given Hospital Id\n")
get_doctors(2)

**Explanation:** This is a manual, application-level join: get_hospital_name() runs its own separate query to look up just the hospital's name (record[1], since Hospital_Name is the second column), and get_doctors() calls it first, then uses that returned name when printing every doctor's record. A real SQL JOIN (SELECT Doctor.*, Hospital.Hospital_Name FROM Doctor JOIN Hospital ON ...) could fetch everything in a single query — the two-query approach shown here is more verbose but keeps each function focused on one clear responsibility, which is sometimes preferred for readability in application code.

## Exercise 5. Calculate and Update a Doctor's Experience

**Concept:** a SELECT to gather data, a calculation in Python, then an UPDATE to write it back — with commit()

**Problem:** Calculate a doctor's years of experience from their joining date, and update their record with that value.

**Given:**
```
doctor_id = 101, joining date 2005-02-10
```

**Expected Output:**
```
Doctor Id: 101  Experience updated to <N> years (N depends on today's date)
```

**Hint:** relativedelta correctly computes whole elapsed years, accounting for whether this year's anniversary of the joining date has passed yet.

In [ ]:
import sqlite3
import datetime
from dateutil.relativedelta import relativedelta

def get_connection():
    return sqlite3.connect('python_db.db')

def close_connection(connection):
    if connection:
        connection.close()

def update_doctor_experience(doctor_id):
    try:
        # Step 1: get the joining date
        connection = get_connection()
        cursor = connection.cursor()
        select_query = """select Joining_Date from Doctor where Doctor_Id = ?"""
        cursor.execute(select_query, (doctor_id,))
        joining_date_row = cursor.fetchone()
        close_connection(connection)

        # Step 2: calculate experience in whole years
        joining_date = datetime.datetime.strptime(joining_date_row[0], '%Y-%m-%d')
        today_date = datetime.datetime.now()
        experience = relativedelta(today_date, joining_date).years

        # Step 3: write the calculated value back to the database
        connection = get_connection()
        cursor = connection.cursor()
        update_query = """update Doctor set Experience = ? where Doctor_Id = ?"""
        cursor.execute(update_query, (experience, doctor_id))
        connection.commit()
        print("Doctor Id:", doctor_id, " Experience updated to", experience, "years")
        close_connection(connection)

    except (Exception, sqlite3.Error) as error:
        print("Error while updating doctor's data", error)

print("Question 5: Calculate and update a doctor's experience\n")

def get_doctor_detail(doctor_id):
    connection = get_connection()
    cursor = connection.cursor()
    cursor.execute("select * from Doctor where Doctor_Id = ?", (doctor_id,))
    row = cursor.fetchone()
    close_connection(connection)
    print(f"  Doctor Id: {row[0]}, Name: {row[1]}, Joining Date: {row[3]}, Experience: {row[6]}")

print("Before:")
get_doctor_detail(101)

update_doctor_experience(101)

print("\nAfter:")
get_doctor_detail(101)

**Explanation:** This exercise combines a read, a calculation done entirely in Python, and a write — three separate steps. datetime.strptime(joining_date_row[0], '%Y-%m-%d') parses the date string retrieved from the database into an actual datetime object. relativedelta(today_date, joining_date).years computes the precise whole-year difference — correctly handling the case where this year's anniversary of the joining date hasn't happened yet, unlike a naive today.year - joining_date.year, which would overcount by one in that situation. Crucially, connection.commit() is called right after the UPDATE executes — without it, the change might never actually persist to the database file, depending on the driver's default transaction handling; this is the single most common mistake when writing data through the DB-API.